In [ ]:
import numpy as np
import cv2
from google.colab.patches import cv2_imshow
import torch
from torch import Tensor

In [ ]:
!pip install fastncut

In [ ]:
from fastncut import ncut, toCosSin

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # 'cpu' # 'cuda'
print(device)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("saeedehkamjoo/standard-test-images")

print("Path to dataset files:", path)

In [ ]:
!ls -l "$path/STI/Old classic/man.bmp"

In [ ]:
img = cv2.imread(f"{path}/STI/Old classic/man.bmp",cv2.IMREAD_GRAYSCALE)

In [ ]:
cv2_imshow(img)

Original algorithm [Shi-Malik 2000]

In [ ]:
def org_ncut(
    intensities: Tensor,
    pe: bool = False,
):
    with torch.no_grad():
        intensities = intensities.float() / 255.0 # 0..1

        dims = intensities.shape
        y = torch.arange(dims[0],device=intensities.device).float()/dims[0]
        x = torch.arange(dims[1],device=intensities.device).float()/dims[1]
        Y, X = torch.meshgrid(y, x, indexing='ij')
        X, Y = X.flatten(), Y.flatten()

        intensities = intensities.flatten() # H*W 0..1
        diff = intensities.unsqueeze(1) - intensities.unsqueeze(0) # shape (H*W, H*W)
        W = torch.exp(-diff**2) # shape (H*W, H*W)

        if pe:
            diffy = Y.unsqueeze(1) - Y.unsqueeze(0)
            diffx = X.unsqueeze(1) - X.unsqueeze(0)
            W *= torch.exp(-(diffx**2+diffy**2)/2)

        eps = 0 #1e-5
        tau = 0.2
        W[W<tau] = eps

        d = W.sum(dim=1) # H*W totals of similarities of one pixel to others
        D = torch.diag(d)

        _, eigenvectors = torch.lobpcg(A=D-W, k=2, B=D, largest=False) # solve the generalized eigenvalue problem
        y1 = eigenvectors[:,1].unsqueeze(1)

        bipartition = y1 > 0

    return bipartition.view(dims)

In [ ]:
trials = 5

In [ ]:
for _ in range(trials):
    tm = cv2.TickMeter()
    tm.start()
    size = 128
    bipartition = org_ncut(torch.tensor(cv2.resize(img,(size,size))).to(device)).cpu().numpy().astype(np.uint8)*255
    tm.stop()
    print('org_ncut',tm.getTimeSec(),'s')

In [ ]:
def fix_polarity(mask):
    border = np.concatenate([mask[0, :], mask[-1, :], mask[:, 0], mask[:, -1]])
    return mask if np.mean(border) > 127 else ~mask

In [ ]:
cv2_imshow(fix_polarity(bipartition))

Our algorithm [Lucny 2026]

In [ ]:
for _ in range(trials):
    tm = cv2.TickMeter()
    tm.start()
    blob = toCosSin(torch.tensor(img,device=device).float().unsqueeze(0)/255.0)
    bipartition = ncut(blob,num_iters=2).cpu().numpy().astype(np.uint8)*255
    tm.stop()
    print('fastncut',tm.getTimeSec(),'s')

In [ ]:
cv2_imshow(fix_polarity(bipartition))